# M4c — Multilabel con grafo (4 experimentos: tabular, texto, grafo, full)

> **El problema, en palabras simples.**
>
> M4 (lineal) y M4b (XGBoost) no separan T1041/T1567 a nivel evento usando texto y campos. Queda una bala: **el contexto**. Un evento aislado no dice mucho, pero lo que pasó en la misma máquina o credencial en 5 minutos sí puede delatar la técnica (bloqueos, flujos, fallos de autenticación alrededor).
>
> Este cuaderno hace la ablación completa: tabular solo (E0), texto+tabular (E1, réplica de M4b), tabular+grafo (E2, nuevo, sin texto) y todo junto (E3 full). Si ni con grafo se separan, la atribución por evento queda cerrada como hint, no como modelo.

> **Síntesis ejecutiva.**
>
> Misma población threat, 6 técnicas y corte temporal que M4/M4b. Cuatro matrices (TAB, TXT+TAB, TAB+G, TXT+TAB+G) × 6 XGBoost binarios con `scale_pos_weight`. Features de grafo: 8 estadísticas del vecindario por host/credencial en ventana de 300 s.
>
> Resultado (`models/m4c_exps.json`): E2 tabular+grafo es el mejor (micro 0.8571, macro 0.2957); E0 tabular (0.8536/0.2643); E3 full (0.7907/0.2779); E1 texto+tabular (0.7926/0.2557). El grafo ayuda (+0.03 de macro sobre E0), el texto lastra. T1041/T1567 siguen sin separarse: ni con grafo.

| Experimento | Matriz | Micro / Macro | Lectura |
|---|---|---|---|
| E0 tab | tabular | 0.8536 / 0.2643 | Baseline fuerte y rapidísimo (3 s) |
| E1 txt+tab | + TF-IDF 5k | 0.7926 / 0.2557 | El texto empeora: sobreajuste al payload |
| E2 tab+grafo | + 8 feats vecindario | 0.8571 / 0.2957 | Mejor: el contexto aporta |
| E3 full | + todo | 0.7907 / 0.2779 | El texto vuelve a lastrar |

> **Fuentes normativas.**
>
> `notebooks/04_m4a_multilabel_logreg.ipynb` (M4), `notebooks/05_m4b_multilabel_xgb.ipynb` (M4b), EDA §12 y M3 (`experiments/m3_grafo/train.py`, agregación causal), `models/m4c_exps.json`, `reports/paper.pdf` §Limitations.
>
> **Reproducción:** celda de grafo lenta (5–10 min, una sola vez) + parrilla lenta (minutos). `uv run jupyter nbconvert --to notebook --execute notebooks/06_m4c_multilabel_grafo.ipynb --output 06_m4c_multilabel_grafo.ipynb --ExecutePreprocessor-timeout=2400`.

---

**Convención de lectura.**

Cada celda de código va precedida por una celda de contexto (qué se hace, por qué, cómo debe leerse y a qué alimenta) y seguida por una celda de observaciones (hecho medido, interpretación y conexión).

La sección final integra los hallazgos y cierra la ablación ATT&CK.



### 1. Por qué probar el grafo y cómo se va a evaluar

**1.1 Problema.**

Texto y tabular agotaron su poder en M4/M4b: T1041/T1567 no se separan y la mejora del micro es espejismo de T1190. La hipótesis restante es estructural: la técnica se revela en la vecindad (ráfaga de bloqueos, flujos o fallos alrededor del evento), no en el evento.

**1.2 Hipótesis.**

- **H4e. El grafo aporta donde el texto no.** E2 superará a E0/E1 en macro-F1 porque las 8 estadísticas de vecindario (fracción de bloqueos, de firewall, de fallos de autenticación, conteos de hosts/usuarios/tipos, duración) capturan lateralidad invisible por fila.

- **H4f. Ni con grafo se separan T1041/T1567.** La macro quedará bajo 0.30: a nivel evento son priors del tier, y el veredicto final será hint, no modelo.

**1.3 Protocolo (idéntico a M4/M4b, cambia solo la matriz).**

1. **Cuatro matrices, un protocolo.** E0/E1/E2/E3 sobre el mismo split y los mismos 6 XGBoost por técnica. E1 replica M4b (control de reproducción).

2. **Advertencia de validez temporal.** Las features de este cuaderno usan ventana de ±300 s (incluyen futuro): miden **cota alta** del aporte del grafo, no número de serve. El serve (`app/main.py`, M3) ya es causal pasado (`[t-300s, t)`). Se declara explícitamente.

3. **Métricas.** Micro/macro-F1, hamming, subset-acc por experimento, más reporte por técnica en el mejor.

**1.4 Criterio de cierre.**

Si E2/E3 no superan macro 0.30, la ablación ATT&CK se cierra con el mismo veredicto que tiers en M1→M2→M3: el contexto ayuda pero no resucita la cola.



### 2. Carga y parseo: misma población threat que la familia

**2.1 Qué se hace.**

Se cargan las columnas de `signals.parquet` más las de entidad (`src_ip`, `dst_ip`, `username`) y se filtra threat con parseo de las 6 técnicas y conteo de positivos.

**2.2 Por qué.**

Las columnas de entidad no existían en M4/M4b: son el precio de entrada del grafo (llaves de vecindario por ORG|valor). Todo lo demás idéntico para comparar.

**2.3 Cómo debe leerse.**

Mismo tamaño de threat y mismos positivos por técnica que M4/M4b. Cualquier diferencia posterior es del grafo, no de la población.

**2.4 Trazabilidad.**

Fija población y Y de las secciones 3–6. Nada del cuaderno 1 se reestima.



In [1]:
import json, time
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
ROOT = ROOT if (ROOT/'pyproject.toml').exists() else ROOT.parent
print('ROOT:', ROOT)
CATS = ['message_type','stream_name','product_name','vendor_name','pipeline','org_id','action','severity','protocol']
GFEATS = ['n_neigh','n_hosts','n_users','frac_block','frac_fw','frac_authfail','n_types','span_min']
TECHS = ['T1190','T1041','T1567','T1204','T1486','T1566']
W = 300
COLS = ['label_binary','timestamp','event_time','message_sanitized','attack_techniques','message_type','stream_name','product_name','vendor_name','pipeline','org_id','action','severity','protocol','src_ip','dst_ip','username']
df = pd.read_parquet(ROOT/'data/signals/signals.parquet', columns=COLS)
th = df[df.label_binary!='benign'].copy().reset_index(drop=True)
print('threat:', th.shape, th.label_binary.value_counts().to_dict())
def parse(v):
    try:
        t = json.loads(v) if v else []
        return [x for x in t if x in TECHS]
    except: return []
Y = np.zeros((len(th), len(TECHS)), dtype=int)
for i,v in enumerate(th.attack_techniques.fillna('')):
    for t in parse(v): Y[i, TECHS.index(t)] = 1
print('positivos por técnica:', {t: int(Y[:,j].sum()) for j,t in enumerate(TECHS)})

ROOT: C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto


threat: (114521, 17) {'suspicious': 106793, 'malicious': 7728}


positivos por técnica: {'T1190': 114507, 'T1041': 7742, 'T1567': 7728, 'T1204': 20, 'T1486': 0, 'T1566': 0}


#### 2.1 Observaciones e insights

**Observación.**

Misma población threat (114 521) y mismos positivos por técnica que M4/M4b, más las tres columnas de entidad para el vecindario.

**Interpretación.**

Comparabilidad garantizada con la familia: el experimento mide el aporte marginal del grafo sobre una base ya medida.

**Conexión.**

Habilita features (sección 3) y vecindario (sección 4).



### 3. Features P1–P3 y partición temporal (idénticos)

**3.1 Qué se hace.**

Derivación de `hour`, `msg_len`, `lag_h` y corte por cuantil 80, idénticos a M4/M4b.

**3.2 Por qué.**

Sin cambios: el contraste E0–E3 debe ser puro de matriz. Reestimar features contaminaría la ablación.

**3.3 Cómo debe leerse.**

Mismos tamaños train/test que M4/M4b (91 617/22 904).

**3.4 Trazabilidad.**

Base de las cuatro matrices de la sección 5.



In [2]:
# features base P1-P3 + split temporal
th['ts'] = pd.to_datetime(th['timestamp'], unit='s', utc=True)
th['hour'] = th.ts.dt.hour
th['msg_len'] = th.message_sanitized.fillna('').str.len()
th['lag_h'] = ((th['timestamp']-th['event_time'])/3600).clip(-1e4,1e4).fillna(-1)
for c in CATS: th[c] = th[c].fillna('').astype(str)
msg = th.message_sanitized.fillna('')
cutoff = th['timestamp'].quantile(0.80)
tr = (th['timestamp'] <= cutoff).to_numpy(); te = ~tr
print(f'train/test: {tr.sum()}/{te.sum()} corte: {pd.to_datetime(cutoff, unit="s", utc=True)}')

train/test: 91617/22904 corte: 2024-07-27 05:13:54.964786053+00:00


#### 3.1 Observaciones e insights

**Observación.**

Mismos train/test que la familia M4. Base tabular lista.

**Interpretación.**

La ablación queda limpia: cuatro matrices, un split, un protocolo.

**Conexión.**

Puente a la sección lenta (vecindario) y a las matrices.



### 4. Vecindario por host y credencial (celda lenta, una sola vez)

**4.1 Qué se hace.**

Índice full-data de 2M ordenado por tiempo por entidad (ORG|src, ORG|dst, ORG|user) y agregación de 8 estadísticas en ventana de 300 s alrededor de cada evento threat (5–10 min, con progreso cada 20 000).

**4.2 Por qué.**

Es la traducción tabular de la idea GraphSAGE (Hamilton et al.): agregar el vecindario en vez de mirar el nodo. Las 8 features (conteo de vecinos, hosts, usuarios, fracciones de bloqueo/firewall/fallos, tipos distintos, duración) resumen la ráfaga alrededor del evento.

**4.3 Cómo debe leerse.**

Distribución de G (muchos eventos sin vecinos → ceros) y tiempo total. Advertencia explícita: ventana de ±300 s incluye futuro, luego E2/E3 son cota alta; el serve causal `[t-300s, t)` rendirá igual o menos.

**4.4 Trazabilidad.**

Produce G, la mitad nueva del experimento. Misma receta de agregación que M3 salvo la ventana (M3 ya es causal).



In [3]:
# grafo: índice full-data 2M (como M3) + agregación solo threat (114k). Celda lenta (~5-10 min), una sola vez.
fcols = ['timestamp','org_id','message_type','action','src_ip','dst_ip','username']
full = pd.read_parquet(ROOT/'data/signals/signals.parquet', columns=fcols)
for c in ['src_ip','dst_ip','username','org_id','message_type','action']:
    full[c] = full[c].fillna('').astype(str)
ts = full['timestamp'].to_numpy()
src = full['src_ip'].to_numpy(); dst = full['dst_ip'].to_numpy(); usr = full['username'].to_numpy()
org = full['org_id'].to_numpy(); mmsg = full['message_type'].to_numpy(); act = full['action'].to_numpy()
# mapa artifact? threat son filas de signals: necesitamos posiciones de threat en full. full está en orden parquet = mismo orden que df.
threat_idx = df[df.label_binary!='benign'].index.to_numpy()
print('threat_idx:', len(threat_idx))
index, ent = {}, {}
pos = pd.Series(np.arange(len(full)))
for c, vals in (('s',src),('d',dst),('u',usr)):
    key = np.char.add(org, np.char.add('|', vals.astype(str)))
    if c=='s': ents=key
    for k,s in pos.groupby(key):
        if k.endswith('|'): continue
        arr = s.to_numpy(); index[(c,k)] = arr[np.argsort(ts[arr])]
print('entidades:', len(index))
Es = np.char.add(org[threat_idx], np.char.add('|', src[threat_idx].astype(str)))
Ed = np.char.add(org[threat_idx], np.char.add('|', dst[threat_idx].astype(str)))
Eu = np.char.add(org[threat_idx], np.char.add('|', usr[threat_idx].astype(str)))
out = np.zeros((len(threat_idx),8), dtype=np.float32)
t0=time.time()
for j,i in enumerate(threat_idx):
    cand=set()
    for c,k in (('s',Es[j]),('d',Ed[j]),('u',Eu[j])):
        if k.endswith('|'): continue
        arr=index.get((c,k))
        if arr is None: continue
        lo=np.searchsorted(ts[arr], ts[i]-W); hi=np.searchsorted(ts[arr], ts[i]+W)
        cand.update(arr[lo:hi].tolist())
    cand.discard(i)
    if cand:
        nb=np.fromiter(cand,dtype=np.int64)
        hosts=set(src[nb].tolist())|set(dst[nb].tolist()); hosts.discard('')
        users=set(usr[nb].tolist()); users.discard('')
        a,m,t=mmsg[nb],act[nb],ts[nb]
        # nota: en full mmsg=message_type, act=action (nombres cruzados arriba, se corrige aquí)
        out[j]=[len(nb),len(hosts),len(users),float(np.mean(act[nb]=='block')),float(np.mean(mmsg[nb]=='firewall_action')),float(np.mean(mmsg[nb]=='auth_failure')),float(len(set(mmsg[nb].tolist()))),float(ts[nb].max()-ts[nb].min())/60.0]
    if (j+1)%20000==0: print(f'{j+1}/{len(threat_idx)} {(time.time()-t0)/60:.1f}min', flush=True)
G = pd.DataFrame(out, columns=GFEATS)
print('G.describe:'); print(G.describe())
print(f'tiempo total {(time.time()-t0)/60:.1f} min')

threat_idx: 114521


entidades: 14773


20000/114521 0.3min


40000/114521 0.4min


60000/114521 0.5min


80000/114521 0.6min


100000/114521 0.7min


G.describe:
             n_neigh        n_hosts        n_users     frac_block  \
count  114521.000000  114521.000000  114521.000000  114521.000000   
mean      851.863770      14.236691       0.010295       0.996665   
std      1735.640869      11.161150       0.382407       0.055968   
min         0.000000       0.000000       0.000000       0.000000   
25%        95.000000       3.000000       0.000000       1.000000   
50%       234.000000      10.000000       0.000000       1.000000   
75%       506.000000      22.000000       0.000000       1.000000   
max      7161.000000      47.000000      47.000000       1.000000   

             frac_fw  frac_authfail        n_types       span_min  
count  114521.000000  114521.000000  114521.000000  114521.000000  
mean        0.828698       0.000367       1.006016       8.480283  
std         0.376375       0.013714       0.116071       2.461203  
min         0.000000       0.000000       0.000000       0.000000  
25%         1.000000      

#### 4.1 Observaciones e insights

**Observación.**

G computado para los 114 521 threat con 8 columnas; fracción relevante sin vecinos (ceros honestos, no imputación).

**Interpretación.**

Los ceros son información (evento aislado) y deben quedarse como ceros: aislarse también describe. La ventana con futuro se declara como cota alta, no como trampa oculta.

**Conexión.**

Habilita E2 y E3 en la sección 5. El costo (minutos, una vez) es amortizable: el índice sirve a futuros análisis.



### 5. Cuatro matrices: TAB, TXT+TAB, TAB+G, FULL

**5.1 Qué se hace.**

One-hot + numéricas (TAB), TF-IDF 5k (TXT) y G disperso (G) combinados en E0/E1/E2/E3 con dimensiones impresas.

**5.2 Por qué.**

La ablación factorial separa el aporte del texto del aporte del grafo: E1−E0 aísla texto, E2−E0 aísla grafo, E3−E2 mide interacción. Sin E0 no hay baseline; sin E1 no hay control de reproducción de M4b.

**5.3 Cómo debe leerse.**

Dimensiones por experimento (el ancho salta con TF-IDF; G añade solo 8 columnas densas). E1 debe replicar a M4b.

**5.4 Trazabilidad.**

Entradas de la parrilla de la sección 6. G barato (8 columnas) frente a TXT caro (5k): el costo también se ablaciona.



In [4]:
# matrices: TAB (one-hot+nums), TXT (tfidf 5k), G (8 feats)
from scipy.sparse import hstack, csr_matrix
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
tab = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', max_categories=30), CATS), ('num','passthrough',['hour','msg_len','lag_h'])])
Atr = tab.fit_transform(th[tr][CATS+['hour','msg_len','lag_h']]); Ate = tab.transform(th[te][CATS+['hour','msg_len','lag_h']])
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True, min_df=5)
Ttr = vec.fit_transform(msg[tr].tolist()); Tte = vec.transform(msg[te].tolist())
Gtr = csr_matrix(G.to_numpy()[tr]); Gte = csr_matrix(G.to_numpy()[te])
EXP = {'E0_tab': (Atr,Ate), 'E1_txt+tab': (hstack([Ttr,Atr]).tocsr(), hstack([Tte,Ate]).tocsr()), 'E2_tab+grafo': (hstack([Atr,Gtr]).tocsr(), hstack([Ate,Gte]).tocsr()), 'E3_txt+tab+grafo': (hstack([Ttr,Atr,Gtr]).tocsr(), hstack([Tte,Ate,Gte]).tocsr())}
for k,(a,b) in EXP.items(): print(k, a.shape, b.shape)

E0_tab (91617, 54) (22904, 54)
E1_txt+tab (91617, 5054) (22904, 5054)
E2_tab+grafo (91617, 62) (22904, 62)
E3_txt+tab+grafo (91617, 5062) (22904, 5062)


#### 5.1 Observaciones e insights

**Observación.**

Cuatro matrices construidas; E1 reproduce el X de M4b por construcción.

**Interpretación.**

El diseño factorial permite atribuir cada décima de macro a texto o a grafo. Las 8 columnas denses de G frente a 5k dispersas de TXT anticipan la lectura de costo.

**Conexión.**

Deja lista la parrilla final (4 experimentos × 6 XGBoost).



### 6. Parrilla 4×6 XGBoost y persistencia

**6.1 Qué se hace.**

Para cada experimento, 6 XGBoost binarios con `scale_pos_weight`, predicciones duras, micro/macro-F1, hamming, subset-acc, reporte por técnica y persistencia en `models/m4c_exps.json`.

**6.2 Por qué.**

Mismo aprendiz y ponderación que M4b en las cuatro matrices: la única variable es X. El reporte por técnica del mejor experimento dirá si el grafo movió T1041/T1567 o solo la cola.

**6.3 Cómo debe leerse.**

Por experimento: micro/macro (la brecha es el diagnóstico), hamming, subset-acc y segundos. E1 debe coincidir con M4b (control). El mejor se lee por técnica.

**6.4 Trazabilidad.**

Produce el artefacto citado en Limitations y cierra la familia M4 (M4/M4b/M4c/M4d).



In [5]:
# 4 experimentos × 6 XGB binarios. Celda lenta (~minutos), una sola vez.
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, hamming_loss, accuracy_score, classification_report
res={}
for name,(Xtr,Xte) in EXP.items():
    t0=time.time(); P=np.zeros_like(Y[te]); skipped=[]
    for j,t in enumerate(TECHS):
        pos=int(Y[tr][:,j].sum()); neg=int(tr.sum())-pos
        if pos<2: skipped.append(t); continue
        clf=XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.08,subsample=0.8,colsample_bytree=0.8,tree_method='hist',objective='binary:logistic',scale_pos_weight=neg/pos,eval_metric='logloss',n_jobs=-1,random_state=42)
        clf.fit(Xtr, Y[tr][:,j])
        P[:,j]=(clf.predict_proba(Xte)[:,1]>=0.5).astype(int)
    micro=round(float(f1_score(Y[te],P,average='micro')),4); macro=round(float(f1_score(Y[te],P,average='macro')),4)
    ham=round(float(hamming_loss(Y[te],P)),4); sub=round(float(accuracy_score(Y[te],P)),4)
    res[name]={'micro_f1':micro,'macro_f1':macro,'hamming':ham,'subset_acc':sub,'skipped':skipped,'secs':int(time.time()-t0)}
    print(f'=== {name} micro={micro} macro={macro} ham={ham} sub={sub} ({res[name]["secs"]}s) ===')
    print(classification_report(Y[te],P,target_names=TECHS,zero_division=0))
print(json.dumps(res,indent=2))
Path(ROOT/'models'/'m4c_exps.json').write_text(json.dumps(res,indent=2))
print('guardado models/m4c_exps.json')

C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


=== E0_tab micro=0.8536 macro=0.2643 ham=0.0577 sub=0.8269 (3s) ===
              precision    recall  f1-score   support

       T1190       1.00      1.00      1.00     22892
       T1041       0.26      0.03      0.06      3733
       T1567       0.24      0.03      0.05      3721
       T1204       0.31      1.00      0.48        10
       T1486       0.00      0.00      0.00         0
       T1566       0.00      0.00      0.00         0

   micro avg       0.97      0.76      0.85     30356
   macro avg       0.30      0.34      0.26     30356
weighted avg       0.82      0.76      0.77     30356
 samples avg       0.99      0.89      0.91     30356



C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


=== E1_txt+tab micro=0.7926 macro=0.2557 ham=0.089 sub=0.7334 (205s) ===
              precision    recall  f1-score   support

       T1190       1.00      1.00      1.00     22892
       T1041       0.08      0.06      0.07      3733
       T1567       0.08      0.06      0.07      3721
       T1204       0.24      1.00      0.39        10
       T1486       0.00      0.00      0.00         0
       T1566       0.00      0.00      0.00         0

   micro avg       0.82      0.77      0.79     30356
   macro avg       0.23      0.35      0.26     30356
weighted avg       0.77      0.77      0.77     30356
 samples avg       0.92      0.90      0.87     30356



C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


=== E2_tab+grafo micro=0.8571 macro=0.2957 ham=0.0562 sub=0.831 (4s) ===
              precision    recall  f1-score   support

       T1190       1.00      1.00      1.00     22892
       T1041       0.35      0.04      0.07      3733
       T1567       0.33      0.03      0.06      3721
       T1204       0.48      1.00      0.65        10
       T1486       0.00      0.00      0.00         0
       T1566       0.00      0.00      0.00         0

   micro avg       0.98      0.76      0.86     30356
   macro avg       0.36      0.35      0.30     30356
weighted avg       0.84      0.76      0.77     30356
 samples avg       0.99      0.90      0.92     30356



=== E3_txt+tab+grafo micro=0.7907 macro=0.2779 ham=0.09 sub=0.7293 (169s) ===
              precision    recall  f1-score   support

       T1190       1.00      1.00      1.00     22892
       T1041       0.08      0.06      0.07      3733
       T1567       0.08      0.06      0.07      3721
       T1204       0.36      1.00      0.53        10
       T1486       0.00      0.00      0.00         0
       T1566       0.00      0.00      0.00         0

   micro avg       0.81      0.77      0.79     30356
   macro avg       0.25      0.35      0.28     30356
weighted avg       0.77      0.77      0.77     30356
 samples avg       0.92      0.90      0.86     30356

{
  "E0_tab": {
    "micro_f1": 0.8536,
    "macro_f1": 0.2643,
    "hamming": 0.0577,
    "subset_acc": 0.8269,
    "skipped": [
      "T1486",
      "T1566"
    ],
    "secs": 3
  },
  "E1_txt+tab": {
    "micro_f1": 0.7926,
    "macro_f1": 0.2557,
    "hamming": 0.089,
    "subset_acc": 0.7334,
    "skipped": [
      "T1

C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


#### 6.1 Observaciones e insights

**Observación.**

E2 tabular+grafo mejor global (micro 0.8571, macro 0.2957, hamming 0.0562, subset-acc 0.831, 4 s); E0 (0.8536/0.2643, 3 s); E3 full (0.7907/0.2779, 169 s); E1 (0.7926/0.2557, 205 s, réplica de M4b).

**Interpretación.**

H4e verificada parcialmente: el grafo aporta (+0.03 de macro sobre E0, mejor en todo). Pero el texto lastra en ambas matrices que lo incluyen (E1 < E0, E3 < E2): el TF-IDF sobreajusta al payload dentro del threat. H4f verificada: macro bajo 0.30 en los cuatro; T1041/T1567 siguen sin separarse.

**Conexión.**

Veredicto: ni con grafo hay atribución por evento desplegable. E2 es el mejor vector de la familia y aun así sirve como evidencia para hints, no como modelo. La ablación ATT&CK queda cerrada igual que tiers en M1→M2→M3.



### 7. Conclusión del cuaderno: el grafo ayuda pero no resucita la cola

#### 7.1 Resultados consolidados

| Experimento | Micro / Macro | Hamming | Segundos |
|---|---|---|---|
| E0 tab | 0.8536 / 0.2643 | 0.0577 | 3 |
| E1 txt+tab | 0.7926 / 0.2557 | 0.0890 | 205 |
| E2 tab+grafo | 0.8571 / 0.2957 | 0.0562 | 4 |
| E3 full | 0.7907 / 0.2779 | 0.0900 | 169 |

#### 7.2 Cadena de hallazgos

- **El contexto aporta.** E2 supera a E0 en todo con 8 columnas baratas: la ráfaga alrededor del evento dice más que el evento.

- **El texto lastra dentro del threat.** Con el tier ya fijado, el payload discrimina peor que el contexto y cuesta 50× más tiempo.

- **La macro no llega a 0.30 en ningún caso.** T1041/T1567 son priors del tier también para el grafo con ventana generosa (cota alta con futuro incluido).

#### 7.3 Plan resultante

**Y** multilabel con X = TAB+G como mejor vector (E2). Métricas en pareja micro/macro. Ventana de serve causal `[t-300s, t)` declarada; los números aquí son cota alta.

#### 7.4 Paso siguiente

`notebooks/m4d_tactics` (misma receta por tácticas TA*) y `notebooks/08_m5_anomalia_iforest.ipynb` (red zero-day). Hints tier→técnicas en consola; sin modelo de atribución desplegado.

*El mejor experimento de la familia confirma el veredicto: medir bien es lo que autoriza a no desplegar.*

